# M12b · Orthogonalité et projections — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m12b_orthogonalite/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m12b_orthogonalite/03_notebook.ipynb`, créée par `python tools/start_chapter.py M12b`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M12b.1 à M12b.12, M12b.18, M12b.19 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M12b.24 | Gram-Schmidt, classique et modifié · `gram_schmidt` | 🔨 | ★★ | 25 |
|  | M12b.25 | La projection orthogonale sur un sous-espace · `project` | 🔨 | ★ | 15 |
|  | M12b.26 | La décomposition QR et les moindres carrés par QR · `qr, least_squares_qr` | 🔨 | ★★ | 25 |
|  | M12b.27 | `np.linalg.qr` et `lstsq` contre ta projection et ta QR : les signes de Q et de R | 📦 | ★ | 15 |
| B | M12b.28 | Prédire : Gram-Schmidt classique sur des vecteurs presque colinéaires, QᵀQ vaut-il encore I ? | 🔮 | ★ | 10 |
|  | M12b.29 | Gram-Schmidt classique contre modifié : l'orthogonalité perdue quand les colonnes sont presque dépendantes | 🔬 | ★★ | 20 |
|  | M12b.30 | Projeter un nuage de ℝ³ sur un plan : les résidus orthogonaux en image | 🎨 | ★ | 20 |
| C | M12b.31 | Chasse à l'erreur : une normalisation oubliée, QQᵀ sur une base non orthonormée, R w = Qᵀy résolu dans le mauvais sens | 🐛 | ★ | 15 |
|  | M12b.32 | Un test de propriétés pour la projection : idempotente, symétrique, résidu orthogonal | 🛠️ | ★ | 15 |
|  | M12b.33 | Défi : des colonnes presque dépendantes, des moindres carrés à 1e−8 près de `lstsq` par l'orthogonalisation de [X \| y] | 🏆 | ★★★ | 30 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12b")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : produit scalaire défini par une matrice, coordonnées dans une base orthonormée, matrice orthogonale, Gram-Schmidt (usuel et pondéré), complément orthogonal, projections et matrice de projection, distance à un sous-espace, moindres carrés et QR, polynômes orthogonaux, une feature résidualisée, un codage sur deux directions.
- Écrire les quatre fonctions de `mymath/orthogonal.py` et les tester, puis comparer avec NumPy.
- Prédire et mesurer l'écart à l'orthogonalité, dessiner une projection, chasser des erreurs, tester des propriétés, résoudre des moindres carrés difficiles.

**Rappel express.** Produit scalaire : bilinéaire, symétrique, défini positif ; $\langle \mathbf{x}, \mathbf{y} \rangle_{\mathbf{M}} = \mathbf{x}^\top\mathbf{M}\mathbf{y}$ ; base orthonormée : $c_i = \langle \mathbf{v}, \mathbf{q}_i \rangle$ ; $\mathbf{Q}^\top\mathbf{Q} = \mathbf{I}$, $\mathbf{Q}^{-1} = \mathbf{Q}^\top$ ; Gram-Schmidt : $\mathbf{v}_k = \mathbf{a}_k - \sum_{j<k} \langle \mathbf{a}_k, \mathbf{q}_j \rangle\,\mathbf{q}_j$, $\mathbf{q}_k = \mathbf{v}_k/\lVert \mathbf{v}_k \rVert$ ; $(\operatorname{Im}\mathbf{A})^\perp = \ker\mathbf{A}^\top$ ; projeté : $\mathbf{Q}\mathbf{Q}^\top\mathbf{v}$ ou $\mathbf{A}^\top\mathbf{A}\,\mathbf{c} = \mathbf{A}^\top\mathbf{v}$ ; $\mathbf{P}^2 = \mathbf{P} = \mathbf{P}^\top$ ; moindres carrés : $\hat{\mathbf{y}}$ = projeté de $\mathbf{y}$ sur $\operatorname{Im}\mathbf{X}$ ; QR : $\mathbf{R}\mathbf{w} = \mathbf{Q}^\top\mathbf{y}$, en remontant.

## Partie 0 · Vérifier tes exercices papier (✏️ M12b.1 à M12b.12, 🔗 M12b.18 et M12b.19)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une liste `[0, -2, 5]` pour un vecteur, une liste de lignes `[[1, 0], [-2, 3]]` pour une matrice, ou `True` / `False`.

Les réponses pas encore remplies affichent ⏳.

**Ex M12b.1 — Un produit scalaire défini par une matrice 2 × 2 : normes, angles, orthogonalité**

In [ ]:
# Ex M12b.1: the values you found on paper
answer_M12b_1a = ...  # a) True or False
answer_M12b_1b = ...  # b) a number
answer_M12b_1c = ...  # c) a number (4 decimals)
answer_M12b_1d = ...  # d) a number (4 decimals)
answer_M12b_1e = ...  # e) a number
answer_M12b_1f = ...  # f) True or False
answer_M12b_1g = ...  # g) a number
answer_M12b_1h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M12b_1a, answer_M12b_1b, answer_M12b_1c, answer_M12b_1d, answer_M12b_1e, answer_M12b_1f, answer_M12b_1g, answer_M12b_1h]):
    wbm.check(f"M12b.1{letter}", answer)

**Ex M12b.2 — Les coordonnées dans une base orthonormée : des produits scalaires au lieu d'un système**

In [ ]:
# Ex M12b.2: the values you found on paper
answer_M12b_2a = ...  # a) a number
answer_M12b_2b = ...  # b) a number
answer_M12b_2c = ...  # c) a list of 4 numbers
answer_M12b_2d = ...  # d) a number
answer_M12b_2e = ...  # e) a list of 4 numbers (1 decimal)
answer_M12b_2f = ...  # f) a list of 4 numbers (1 decimal)

for letter, answer in zip("abcdef", [answer_M12b_2a, answer_M12b_2b, answer_M12b_2c, answer_M12b_2d, answer_M12b_2e, answer_M12b_2f]):
    wbm.check(f"M12b.2{letter}", answer)

**Ex M12b.3 — Une matrice orthogonale 3 × 3 conserve normes et angles**

In [ ]:
# Ex M12b.3: the values you found on paper
answer_M12b_3a = ...  # a) True or False
answer_M12b_3b = ...  # b) a list of 3 numbers (4 decimals)
answer_M12b_3c = ...  # c) a number (4 decimals)
answer_M12b_3d = ...  # d) a number
answer_M12b_3e = ...  # e) a list of 3 numbers
answer_M12b_3f = ...  # f) True or False
answer_M12b_3g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M12b_3a, answer_M12b_3b, answer_M12b_3c, answer_M12b_3d, answer_M12b_3e, answer_M12b_3f, answer_M12b_3g]):
    wbm.check(f"M12b.3{letter}", answer)

**Ex M12b.4 — Gram-Schmidt sur trois vecteurs de ℝ³**

In [ ]:
# Ex M12b.4: the values you found on paper
answer_M12b_4a = ...  # a) a number
answer_M12b_4b = ...  # b) a number
answer_M12b_4c = ...  # c) a list of 3 numbers (4 decimals)
answer_M12b_4d = ...  # d) a number
answer_M12b_4e = ...  # e) a list of 3 numbers (4 decimals)
answer_M12b_4f = ...  # f) a list of 2 numbers
answer_M12b_4g = ...  # g) a list of 3 numbers (4 decimals)
answer_M12b_4h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M12b_4a, answer_M12b_4b, answer_M12b_4c, answer_M12b_4d, answer_M12b_4e, answer_M12b_4f, answer_M12b_4g, answer_M12b_4h]):
    wbm.check(f"M12b.4{letter}", answer)

**Ex M12b.5 — Gram-Schmidt pour un produit scalaire qui n'est pas l'usuel**

In [ ]:
# Ex M12b.5: the values you found on paper
answer_M12b_5a = ...  # a) a number
answer_M12b_5b = ...  # b) a number
answer_M12b_5c = ...  # c) a list of 3 numbers (1 decimal)
answer_M12b_5d = ...  # d) a list of 3 numbers (1 decimal)
answer_M12b_5e = ...  # e) a number
answer_M12b_5f = ...  # f) a list of 3 numbers (4 decimals)
answer_M12b_5g = ...  # g) a list of 2 numbers
answer_M12b_5h = ...  # h) a list of 3 numbers

for letter, answer in zip("abcdefgh", [answer_M12b_5a, answer_M12b_5b, answer_M12b_5c, answer_M12b_5d, answer_M12b_5e, answer_M12b_5f, answer_M12b_5g, answer_M12b_5h]):
    wbm.check(f"M12b.5{letter}", answer)

**Ex M12b.6 — Une base du complément orthogonal d'un plan de ℝ⁴**

In [ ]:
# Ex M12b.6: the values you found on paper
answer_M12b_6a = ...  # a) an integer
answer_M12b_6b = ...  # b) a list of 4 numbers
answer_M12b_6c = ...  # c) a list of 4 numbers
answer_M12b_6d = ...  # d) True or False
answer_M12b_6e = ...  # e) a list of 2 numbers
answer_M12b_6f = ...  # f) True or False
answer_M12b_6g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M12b_6a, answer_M12b_6b, answer_M12b_6c, answer_M12b_6d, answer_M12b_6e, answer_M12b_6f, answer_M12b_6g]):
    wbm.check(f"M12b.6{letter}", answer)

**Ex M12b.7 — Projeter sur un plan de ℝ³ : avec une base quelconque, puis orthonormée**

In [ ]:
# Ex M12b.7: the values you found on paper
answer_M12b_7a = ...  # a) a matrix
answer_M12b_7b = ...  # b) a list of 2 numbers
answer_M12b_7c = ...  # c) a list of 2 numbers
answer_M12b_7d = ...  # d) a list of 3 numbers
answer_M12b_7e = ...  # e) a list of 2 numbers
answer_M12b_7f = ...  # f) a list of 3 numbers
answer_M12b_7g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M12b_7a, answer_M12b_7b, answer_M12b_7c, answer_M12b_7d, answer_M12b_7e, answer_M12b_7f, answer_M12b_7g]):
    wbm.check(f"M12b.7{letter}", answer)

**Ex M12b.8 — La matrice de projection : P² = P, Pᵀ = P, trace et rang**

In [ ]:
# Ex M12b.8: the values you found on paper
answer_M12b_8a = ...  # a) True or False
answer_M12b_8b = ...  # b) a list of 3 numbers
answer_M12b_8c = ...  # c) a list of 3 numbers
answer_M12b_8d = ...  # d) a number
answer_M12b_8e = ...  # e) an integer
answer_M12b_8f = ...  # f) a list of 3 numbers
answer_M12b_8g = ...  # g) True or False
answer_M12b_8h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M12b_8a, answer_M12b_8b, answer_M12b_8c, answer_M12b_8d, answer_M12b_8e, answer_M12b_8f, answer_M12b_8g, answer_M12b_8h]):
    wbm.check(f"M12b.8{letter}", answer)

**Ex M12b.9 — La distance d'un point à un sous-espace, et le résidu orthogonal**

In [ ]:
# Ex M12b.9: the values you found on paper
answer_M12b_9a = ...  # a) True or False
answer_M12b_9b = ...  # b) a number
answer_M12b_9c = ...  # c) a number
answer_M12b_9d = ...  # d) a list of 4 numbers
answer_M12b_9e = ...  # e) a list of 4 numbers
answer_M12b_9f = ...  # f) a number
answer_M12b_9g = ...  # g) a number (4 decimals)
answer_M12b_9h = ...  # h) a number

for letter, answer in zip("abcdefgh", [answer_M12b_9a, answer_M12b_9b, answer_M12b_9c, answer_M12b_9d, answer_M12b_9e, answer_M12b_9f, answer_M12b_9g, answer_M12b_9h]):
    wbm.check(f"M12b.9{letter}", answer)

**Ex M12b.10 — Des moindres carrés comme une projection : y, son projeté et le résidu**

In [ ]:
# Ex M12b.10: the values you found on paper
answer_M12b_10a = ...  # a) a matrix
answer_M12b_10b = ...  # b) a list of 2 numbers
answer_M12b_10c = ...  # c) a list of 2 numbers (1 decimal)
answer_M12b_10d = ...  # d) a list of 5 numbers (1 decimal)
answer_M12b_10e = ...  # e) a list of 5 numbers (1 decimal)
answer_M12b_10f = ...  # f) a list of 2 numbers
answer_M12b_10g = ...  # g) a number (1 decimal)
answer_M12b_10h = ...  # h) a number (1 decimal)

for letter, answer in zip("abcdefgh", [answer_M12b_10a, answer_M12b_10b, answer_M12b_10c, answer_M12b_10d, answer_M12b_10e, answer_M12b_10f, answer_M12b_10g, answer_M12b_10h]):
    wbm.check(f"M12b.10{letter}", answer)

**Ex M12b.11 — La décomposition QR d'une matrice 3 × 2, puis les moindres carrés par R w = Qᵀy**

In [ ]:
# Ex M12b.11: the values you found on paper
answer_M12b_11a = ...  # a) a number
answer_M12b_11b = ...  # b) a number
answer_M12b_11c = ...  # c) a number
answer_M12b_11d = ...  # d) a list of 3 numbers (4 decimals)
answer_M12b_11e = ...  # e) a matrix
answer_M12b_11f = ...  # f) a list of 2 numbers
answer_M12b_11g = ...  # g) a list of 2 numbers (4 decimals)
answer_M12b_11h = ...  # h) a list of 3 numbers

for letter, answer in zip("abcdefgh", [answer_M12b_11a, answer_M12b_11b, answer_M12b_11c, answer_M12b_11d, answer_M12b_11e, answer_M12b_11f, answer_M12b_11g, answer_M12b_11h]):
    wbm.check(f"M12b.11{letter}", answer)

**Ex M12b.12 — Le produit scalaire de fonctions sur [−1 ; 1] : orthogonaliser 1, x et x²**

In [ ]:
# Ex M12b.12: the values you found on paper
answer_M12b_12a = ...  # a) a number
answer_M12b_12b = ...  # b) a number (4 decimals)
answer_M12b_12c = ...  # c) a number
answer_M12b_12d = ...  # d) a number (4 decimals)
answer_M12b_12e = ...  # e) a number (4 decimals)
answer_M12b_12f = ...  # f) a number (4 decimals)
answer_M12b_12g = ...  # g) a list of 3 numbers (4 decimals)

for letter, answer in zip("abcdefg", [answer_M12b_12a, answer_M12b_12b, answer_M12b_12c, answer_M12b_12d, answer_M12b_12e, answer_M12b_12f, answer_M12b_12g]):
    wbm.check(f"M12b.12{letter}", answer)

**Ex M12b.18 — Pont : retirer d'une feature ce qu'explique une autre, une projection puis un résidu**

In [ ]:
# Ex M12b.18: the values you found on paper
answer_M12b_18a = ...  # a) a number
answer_M12b_18b = ...  # b) a list of 4 numbers
answer_M12b_18c = ...  # c) a list of 4 numbers
answer_M12b_18d = ...  # d) a number
answer_M12b_18e = ...  # e) a number (4 decimals)
answer_M12b_18f = ...  # f) a number (2 decimals)
answer_M12b_18g = ...  # g) a list of 2 numbers (2 decimals)

for letter, answer in zip("abcdefg", [answer_M12b_18a, answer_M12b_18b, answer_M12b_18c, answer_M12b_18d, answer_M12b_18e, answer_M12b_18f, answer_M12b_18g]):
    wbm.check(f"M12b.18{letter}", answer)

**Ex M12b.19 — Pont : projeter sur k directions orthonormées, reconstruire, et l'erreur par Pythagore**

In [ ]:
# Ex M12b.19: the values you found on paper
answer_M12b_19a = ...  # a) True or False
answer_M12b_19b = ...  # b) a list of 2 numbers
answer_M12b_19c = ...  # c) a list of 4 numbers (1 decimal)
answer_M12b_19d = ...  # d) a list of 4 numbers (1 decimal)
answer_M12b_19e = ...  # e) a list of 2 numbers
answer_M12b_19f = ...  # f) a number (2 decimals)
answer_M12b_19g = ...  # g) a list of 2 numbers
answer_M12b_19h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M12b_19a, answer_M12b_19b, answer_M12b_19c, answer_M12b_19d, answer_M12b_19e, answer_M12b_19f, answer_M12b_19g, answer_M12b_19h]):
    wbm.check(f"M12b.19{letter}", answer)

## Partie A · Ta librairie : Gram-Schmidt, projection, QR et moindres carrés ; NumPy

Fiche §3, §5 et §7. Les quatre fonctions de `mymath/orthogonal.py`, puis les mêmes calculs avec NumPy.

In [ ]:
# Tools of the notebook (parts A to C)
import matplotlib.pyplot as plt
import numpy as np

from wbm.nbtools import StopCheck, checked, drawn, run_pytest_group


def raised(function, *args, **kwargs):
    """The name of the exception that function(*args, **kwargs) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        function(*args, **kwargs)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"


def orthogonality_loss(Q):
    """The largest |QᵀQ − I|: 0 for orthonormal columns, about 1 when two columns point the same way."""
    Q = np.asarray(Q, dtype=float)
    return float(np.max(np.abs(Q.T @ Q - np.eye(Q.shape[1]))))

### Ex M12b.24 — Gram-Schmidt, classique et modifié · `gram_schmidt` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer le procédé de Gram-Schmidt, dans ses versions classique et modifiée, avec la convention de signe et la détection des colonnes dépendantes.  
**Prérequis :** fiche §3 · **mymath :** `orthogonal.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/orthogonal.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Une famille de vecteurs est donnée par les **colonnes** d'une matrice. NumPy est permis, `@`, `.T`, `np.linalg.norm` et `np.linalg.solve` compris ; `np.linalg.qr`, `lstsq`, `pinv`, `inv`, `scipy.linalg` et scikit-learn sont interdits : les tests comparent tes résultats aux leurs, et un test vérifie que tu ne t'en sers pas. Ne modifie jamais la matrice reçue : travaille sur une copie, `np.array(A, dtype=float)`.

Écris `gram_schmidt(A, modified=True, tol=1e-10)` (fiche §3), en suivant la docstring : une copie en flottants, une `ValueError` si $\mathbf{A}$ n'est pas à deux dimensions, puis une boucle sur les colonnes. Pour la colonne $k$ : $\mathbf{v} \leftarrow \mathbf{a}_k$, puis, pour chaque $j < k$, $\mathbf{v} \leftarrow \mathbf{v} - c\,\mathbf{q}_j$ avec $c = \mathbf{q}_j \cdot \mathbf{a}_k$ (classique, `modified=False`) ou $c = \mathbf{q}_j \cdot \mathbf{v}$ (modifié, le vecteur **courant**). Si $\lVert \mathbf{v} \rVert \le \mathrm{tol} \times \lVert \mathbf{a}_k \rVert$ (ou si $\mathbf{v}$ est nul), les colonnes sont dépendantes : `ValueError`. Sinon $\mathbf{q}_k = \mathbf{v}/\lVert \mathbf{v} \rVert$, ce qui donne $\mathbf{q}_k \cdot \mathbf{a}_k > 0$.

La vérification calcule, avec $\mathbf{A}$ = `[[6, 3], [6, 2], [7, 2]]` :
- a) `gram_schmidt(A)` (une matrice, 6 décimales) ;
- b) vrai ou faux : `gram_schmidt(A, modified=False)` et `gram_schmidt(A)` sont égales à $10^{-12}$ près ;
- c) vrai ou faux : pour une matrice de forme $(6, 4)$ tirée au hasard (`np.random.default_rng(24)`, loi normale), `orthogonality_loss(gram_schmidt(...))` est inférieure à $10^{-12}$ ;
- d) `gram_schmidt([[-2, 0], [0, -3]])` (une matrice) ;
- e) le nom de l'erreur levée par `gram_schmidt([[1, 2], [2, 4], [3, 6]])` ;
- f) le nom de l'erreur levée par `gram_schmidt([1, 2, 3])` ;
- g) vrai ou faux : après `gram_schmidt(M)`, le tableau `M` (une copie en flottants de $\mathbf{A}$) n'a pas changé.

La fonction `orthogonality_loss(Q)` des outils rend le plus grand $|(\mathbf{Q}^\top\mathbf{Q} - \mathbf{I})_{ij}|$. Dans ta copie : retrouve a) à la main (les colonnes de $\mathbf{Q}$ sont des onzièmes). Pourquoi d) a-t-il des $-1$ sur la diagonale ? Pourquoi e) lève-t-elle une erreur, et que vaut $\mathbf{v}_2$ dans ce cas ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12b")  # reload your saved file

In [ ]:
M_24 = np.array([[6, 3], [6, 2], [7, 2]], dtype=float)
with checked("M12b.24a"):
    wbm.check("M12b.24a", np.asarray(mymath.orthogonal.gram_schmidt([[6, 3], [6, 2], [7, 2]])).tolist(), computed=True)
with checked("M12b.24b"):
    wbm.check("M12b.24b", bool(np.allclose(mymath.orthogonal.gram_schmidt([[6, 3], [6, 2], [7, 2]], modified=False),
                                           mymath.orthogonal.gram_schmidt([[6, 3], [6, 2], [7, 2]]), rtol=0, atol=1e-12)), computed=True)
with checked("M12b.24c"):
    wbm.check("M12b.24c", orthogonality_loss(mymath.orthogonal.gram_schmidt(
        np.random.default_rng(24).normal(size=(6, 4)))) < 1e-12, computed=True)
with checked("M12b.24d"):
    wbm.check("M12b.24d", np.asarray(mymath.orthogonal.gram_schmidt([[-2, 0], [0, -3]])).tolist(), computed=True)
with checked("M12b.24e"):
    wbm.check("M12b.24e", raised(mymath.orthogonal.gram_schmidt, [[1, 2], [2, 4], [3, 6]]), computed=True)
with checked("M12b.24f"):
    wbm.check("M12b.24f", raised(mymath.orthogonal.gram_schmidt, [1, 2, 3]), computed=True)
with checked("M12b.24g"):
    mymath.orthogonal.gram_schmidt(M_24)
    wbm.check("M12b.24g", bool(np.array_equal(M_24, np.array([[6, 3], [6, 2], [7, 2]], dtype=float))), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m12b_orthogonal.py", "gram_schmidt or test_orthogonal_uses_")

### Ex M12b.25 — La projection orthogonale sur un sous-espace · `project` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer la projection orthogonale sur l'espace des colonnes d'une matrice, par une base orthonormée.  
**Prérequis :** fiche §5 ; Ex M12b.24 · **mymath :** `orthogonal.py` · **Parcours :** C, E

Écris `project(v, A)` (fiche §5), en suivant la docstring : vérifie que $\mathbf{v}$ est à une dimension, $\mathbf{A}$ à deux, et que $\mathbf{A}$ a autant de lignes que $\mathbf{v}$ a de coordonnées (sinon `ValueError`) ; puis $\mathbf{Q}$ = `gram_schmidt(A)` (qui lève elle-même la `ValueError` des colonnes dépendantes), et $\mathbf{p} = \mathbf{Q}(\mathbf{Q}^\top\mathbf{v})$. Pas d'équations normales ici : M09b les a déjà programmées.

La vérification calcule :
- a) `project([2, 0, 1, 5], [[1, 0], [1, 1], [1, 2], [1, 3]])` (une liste, 6 décimales) ;
- b) `project([4, 3], [[1], [3]])` (une liste, 6 décimales) ;
- c) vrai ou faux : pour une matrice $\mathbf{A}$ de forme $(6, 3)$ et un vecteur $\mathbf{v}$ tirés au hasard (`np.random.default_rng(25)`), projeter une seconde fois $\mathbf{p}$ = `project(v, A)` redonne $\mathbf{p}$, à $10^{-12}$ près ;
- d) vrai ou faux : pour les mêmes, $\mathbf{A}^\top(\mathbf{v} - \mathbf{p})$ est nul à $10^{-10}$ près ;
- e) vrai ou faux : `project(v, A @ M)`, avec une matrice $\mathbf{M}$ de forme $(3, 3)$ tirée au hasard (inversible), redonne $\mathbf{p}$ à $10^{-10}$ près ;
- f) le nom de l'erreur levée par `project([1, 2], [[1, 0], [0, 1], [0, 0]])` ;
- g) le nom de l'erreur levée par `project([1, 2, 3], [[1, 2], [1, 2], [1, 2]])`.

Dans ta copie : quelle droite de régression a) calcule-t-il, et que représentent les quatre nombres ? Vérifie b) avec la formule de la projection sur une droite (M07). Pourquoi e) était-il prévisible ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12b")  # reload your saved file

In [ ]:
rng_25 = np.random.default_rng(25)
A_25, v_25, M_25 = rng_25.normal(size=(6, 3)), rng_25.normal(size=6), rng_25.normal(size=(3, 3))
with checked("M12b.25a"):
    wbm.check("M12b.25a", np.asarray(mymath.orthogonal.project([2, 0, 1, 5], [[1, 0], [1, 1], [1, 2], [1, 3]])).tolist(),
              computed=True)
with checked("M12b.25b"):
    wbm.check("M12b.25b", np.asarray(mymath.orthogonal.project([4, 3], [[1], [3]])).tolist(), computed=True)
with checked("M12b.25c"):
    p_25 = mymath.orthogonal.project(v_25, A_25)
    wbm.check("M12b.25c", bool(np.allclose(mymath.orthogonal.project(p_25, A_25), p_25, rtol=0, atol=1e-12)),
              computed=True)
with checked("M12b.25d"):
    p_25 = mymath.orthogonal.project(v_25, A_25)
    wbm.check("M12b.25d", bool(np.allclose(A_25.T @ (v_25 - p_25), 0, rtol=0, atol=1e-10)), computed=True)
with checked("M12b.25e"):
    p_25 = mymath.orthogonal.project(v_25, A_25)
    wbm.check("M12b.25e", bool(np.allclose(mymath.orthogonal.project(v_25, A_25 @ M_25), p_25, rtol=0, atol=1e-10)),
              computed=True)
with checked("M12b.25f"):
    wbm.check("M12b.25f", raised(mymath.orthogonal.project, [1, 2], [[1, 0], [0, 1], [0, 0]]), computed=True)
with checked("M12b.25g"):
    wbm.check("M12b.25g", raised(mymath.orthogonal.project, [1, 2, 3], [[1, 2], [1, 2], [1, 2]]), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m12b_orthogonal.py", "test_project_ or test_orthogonal_uses_")

### Ex M12b.26 — La décomposition QR et les moindres carrés par QR · `qr, least_squares_qr` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer la décomposition QR à partir de Gram-Schmidt, puis les moindres carrés par un système triangulaire.  
**Prérequis :** fiche §7 ; Ex M12b.24 · **mymath :** `orthogonal.py` · **Parcours :** C

Écris, en suivant les docstrings :
- `qr(A)` (fiche §7) : $\mathbf{Q}$ = `gram_schmidt(A)` (modifié), puis $\mathbf{R} = \mathbf{Q}^\top\mathbf{A}$, dont tu mets **exactement** à 0 les éléments sous la diagonale (`np.triu`) ; rends le couple `(Q, R)` ;
- `least_squares_qr(X, y)` (fiche §7) : vérifie les formes (`ValueError`), calcule `qr(X)`, le second membre $\mathbf{Q}^\top\mathbf{y}$, puis résous $\mathbf{R}\mathbf{w} = \mathbf{Q}^\top\mathbf{y}$ **en remontant** depuis la dernière ligne : $w_i = \big(z_i - \sum_{j > i} r_{ij}w_j\big)/r_{ii}$. Ni `np.linalg.solve` ni `inv` : la boucle.

La vérification calcule :
- a) la matrice $\mathbf{R}$ de `qr([[12, 6], [3, 2], [4, 0]])` (6 décimales) ;
- b) vrai ou faux : pour cette matrice, $\mathbf{Q}\mathbf{R}$ redonne $\mathbf{A}$ à $10^{-12}$ près ;
- c) vrai ou faux : pour une matrice de forme $(5, 3)$ tirée au hasard (`np.random.default_rng(26)`), les éléments de $\mathbf{R}$ sous la diagonale sont **exactement** nuls ;
- d) `least_squares_qr([[1, 0], [1, 1], [1, 2], [1, 3]], [1, 3, 2, 6])` (une liste, 6 décimales) ;
- e) vrai ou faux : pour $\mathbf{X}$ de forme $(30, 4)$ et $\mathbf{y}$ tirés au hasard (`np.random.default_rng(26)`, après la matrice de c), `least_squares_qr(X, y)` et `np.linalg.solve(X.T @ X, X.T @ y)` sont égaux à $10^{-9}$ près ;
- f) le nom de l'erreur levée par `least_squares_qr([[1, 0], [0, 1]], [1, 2, 3])` ;
- g) le nom de l'erreur levée par `qr([[1, 2], [2, 4]])`.

Dans ta copie : retrouve a) à la main (les colonnes de $\mathbf{Q}$ sont des treizièmes). Pour d), écris $\mathbf{R}$ et $\mathbf{Q}^\top\mathbf{y}$, puis résous à la main. Pourquoi c) demande-t-il `np.triu`, alors que $\mathbf{Q}^\top\mathbf{A}$ est « déjà » triangulaire ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12b")  # reload your saved file

In [ ]:
rng_26 = np.random.default_rng(26)
B_26 = rng_26.normal(size=(5, 3))
X_26r, y_26r = rng_26.normal(size=(30, 4)), rng_26.normal(size=30)
with checked("M12b.26a"):
    wbm.check("M12b.26a", np.asarray(mymath.orthogonal.qr([[12, 6], [3, 2], [4, 0]])[1]).tolist(), computed=True)
with checked("M12b.26b"):
    Q_26, R_26 = mymath.orthogonal.qr([[12, 6], [3, 2], [4, 0]])
    wbm.check("M12b.26b", bool(np.allclose(Q_26 @ R_26, [[12, 6], [3, 2], [4, 0]], rtol=0, atol=1e-12)), computed=True)
with checked("M12b.26c"):
    wbm.check("M12b.26c", bool(np.all(np.tril(mymath.orthogonal.qr(B_26)[1], -1) == 0)), computed=True)
with checked("M12b.26d"):
    wbm.check("M12b.26d", np.asarray(mymath.orthogonal.least_squares_qr([[1, 0], [1, 1], [1, 2], [1, 3]], [1, 3, 2, 6])).tolist(), computed=True)
with checked("M12b.26e"):
    wbm.check("M12b.26e", bool(np.allclose(mymath.orthogonal.least_squares_qr(X_26r, y_26r),
                                           np.linalg.solve(X_26r.T @ X_26r, X_26r.T @ y_26r), rtol=0, atol=1e-9)),
              computed=True)
with checked("M12b.26f"):
    wbm.check("M12b.26f", raised(mymath.orthogonal.least_squares_qr, [[1, 0], [0, 1]], [1, 2, 3]), computed=True)
with checked("M12b.26g"):
    wbm.check("M12b.26g", raised(mymath.orthogonal.qr, [[1, 2], [2, 4]]), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m12b_orthogonal.py", "test_qr_ or test_least_squares_qr_ or test_orthogonal_uses_")

### Ex M12b.27 — `np.linalg.qr` et `lstsq` contre ta projection et ta QR : les signes de Q et de R 📦 ★ ⏱️ 15 min
**Objectif :** utiliser `np.linalg.qr` et `np.linalg.lstsq`, lire ce qu'elles rendent, et comparer avec ta librairie au signe près.  
**Prérequis :** fiche §7 ; Ex M12b.26 · **Parcours :** C

En pratique, on ne réécrit pas ces fonctions :
- `np.linalg.qr(A)` rend la décomposition QR réduite `(Q, R)`, calculée par des réflexions de Householder (LAPACK) ; `np.linalg.qr(A, mode="complete")` rend la forme complète. Les signes ne suivent pas la convention $r_{kk} > 0$ ;
- `np.linalg.lstsq(X, y, rcond=None)` rend **quatre** objets : les poids, la somme des carrés des résidus (dans un tableau, vide si le rang n'est pas plein ou s'il y a autant de colonnes que de lignes), le rang de $\mathbf{X}$, et ses valeurs singulières (M12d). Elle passe par la SVD, et accepte des colonnes dépendantes.

Avec $\mathbf{A}$ = `[[2, -1, 0], [1, 3, -2], [0, 1, 4], [-3, 2, 1], [1, 0, 2]]` et $\mathbf{y}$ = `[1, 0, 2, -1, 3]`, calcule :
- a) les formes de $\mathbf{Q}$ et de $\mathbf{R}$ rendues par `np.linalg.qr(A)` (une liste de deux formes, `[[.., ..], [.., ..]]`) ;
- b) la forme de $\mathbf{Q}$ avec `mode="complete"` (une liste de 2 entiers) ;
- c) vrai ou faux : les valeurs absolues des $\mathbf{R}$ de NumPy et de ta fonction `qr` sont égales à $10^{-10}$ près ;
- d) vrai ou faux : avec $\mathbf{D}$ = `np.diag(np.sign(np.diag(R_numpy)))`, `Q_numpy @ D` et `D @ R_numpy` sont égales à ton $\mathbf{Q}$ et à ton $\mathbf{R}$, à $10^{-10}$ près ;
- e) le rang rendu par `np.linalg.lstsq(A, y, rcond=None)` ;
- f) vrai ou faux : les poids de `lstsq` et ceux de ta fonction `least_squares_qr(A, y)` sont égaux à $10^{-10}$ près ;
- g) le rang rendu par `lstsq` pour la matrice $\mathbf{A}_4$ = `np.column_stack([A, A[:, 0] + A[:, 1]])` (quatre colonnes) ;
- h) le nom de l'erreur levée par ta fonction `least_squares_qr(A_4, y)`.

Dans ta copie : pourquoi les valeurs absolues de $\mathbf{R}$ sont-elles les mêmes alors que les signes diffèrent ? Que vérifie d) ? Pourquoi `lstsq` accepte-t-elle $\mathbf{A}_4$, et que peut-elle bien rendre, puisqu'une infinité de poids conviennent (M12a §6) ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12b")
A_27 = np.array([[2, -1, 0], [1, 3, -2], [0, 1, 4], [-3, 2, 1], [1, 0, 2]], dtype=float)
y_27 = np.array([1, 0, 2, -1, 3], dtype=float)
A4_27 = np.column_stack([A_27, A_27[:, 0] + A_27[:, 1]])
answer_M12b_27a = ...  # a) [shape of Q, shape of R]
answer_M12b_27b = ...  # b) a list of 2 integers
answer_M12b_27e = ...  # e) an integer
answer_M12b_27g = ...  # g) an integer
answer_M12b_27c = answer_M12b_27d = answer_M12b_27f = answer_M12b_27h = None
with wbm.attempt("M12b.27c"):        # your qr is called here: ⏳ while it is not written
    answer_M12b_27c = ...  # c) True or False
with wbm.attempt("M12b.27d"):
    answer_M12b_27d = ...  # d) True or False
with wbm.attempt("M12b.27f"):        # your least_squares_qr is called here
    answer_M12b_27f = ...  # f) True or False
with wbm.attempt("M12b.27h"):
    answer_M12b_27h = ...  # h) the name of the error: raised(mymath.orthogonal.least_squares_qr, A4_27, y_27)

In [ ]:
for letter, answer in zip("abcdefgh", [answer_M12b_27a, answer_M12b_27b, answer_M12b_27c, answer_M12b_27d,
                                       answer_M12b_27e, answer_M12b_27f, answer_M12b_27g, answer_M12b_27h]):
    wbm.check(f"M12b.27{letter}", answer)

## Partie B · Une prédiction, une expérience et une figure

Fiche §3, §5 et §8. Gram-Schmidt sur des colonnes presque colinéaires, l'écart à l'orthogonalité des deux procédés, un nuage projeté sur un plan.

### Ex M12b.28 — Prédire : Gram-Schmidt classique sur des vecteurs presque colinéaires, QᵀQ vaut-il encore I ? 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir ce que deviennent l'orthogonalité et les normes calculées par Gram-Schmidt quand les colonnes sont presque colinéaires, puis le vérifier.  
**Prérequis :** fiche §3 ; Ex M12b.24 · **Parcours :** C, E

La cellule fournie fabrique $\mathbf{A}$, de forme $(5, 3)$, dont les trois colonnes sont un même vecteur plus des écarts de l'ordre de $10^{-9}$ : libres, mais presque colinéaires. $\mathbf{B}$, de forme $(50, 3)$, a des colonnes tirées au hasard, bien séparées. L'« écart à l'orthogonalité » est le plus grand $|(\mathbf{Q}^\top\mathbf{Q} - \mathbf{I})_{ij}|$ (la fonction `orthogonality_loss` des outils). Pour chaque question, choisis le numéro de la bonne réponse :
- a) l'écart de Gram-Schmidt **classique** sur $\mathbf{A}$ : (1) de l'ordre de $10^{-15}$ ; (2) entre $10^{-12}$ et $10^{-4}$ ; (3) au-dessus de $0{,}1$ ;
- b) l'écart de Gram-Schmidt **modifié** sur $\mathbf{A}$ : mêmes choix ;
- c) l'écart du **classique** sur $\mathbf{B}$ : mêmes choix ;
- d) les normes des colonnes de $\mathbf{Q}$ (classique, sur $\mathbf{A}$) : (1) toutes égales à 1 à $10^{-12}$ près ; (2) certaines loin de 1 ; (3) certaines nulles.

**Avant d'exécuter quoi que ce soit**, écris tes quatre prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience.

In [ ]:
rng_28 = np.random.default_rng(28)
base_28 = rng_28.normal(size=5)
A_28 = base_28[:, None] + 1e-9 * rng_28.normal(size=(5, 3))     # three columns that differ by about 1e-9
B_28 = rng_28.normal(size=(50, 3))                               # three columns well apart

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M12b_28a = ...  # a) 1, 2 or 3, written BEFORE running the experiment below
prediction_M12b_28b = ...  # b) 1, 2 or 3
prediction_M12b_28c = ...  # c) 1, 2 or 3
prediction_M12b_28d = ...  # d) 1, 2 or 3

In [ ]:
for letter, prediction in zip("abcd", [prediction_M12b_28a, prediction_M12b_28b, prediction_M12b_28c,
                                      prediction_M12b_28d]):
    wbm.check(f"M12b.28{letter}", prediction)

**L'expérience** (avec ta fonction `gram_schmidt`, ou la référence si elle n'est pas encore écrite). Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M12b_28a, prediction_M12b_28b, prediction_M12b_28c, prediction_M12b_28d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12b")
    Q_classical = mymath.orthogonal.gram_schmidt(A_28, modified=False)
    Q_modified = mymath.orthogonal.gram_schmidt(A_28)
    print(f"a) classique, colonnes presque colinéaires : écart {orthogonality_loss(Q_classical):.1e}")
    print(f"b) modifié, mêmes colonnes                : écart {orthogonality_loss(Q_modified):.1e}")
    print(f"c) classique, colonnes bien séparées      : écart "
          f"{orthogonality_loss(mymath.orthogonal.gram_schmidt(B_28, modified=False)):.1e}")
    print(f"d) normes des colonnes (classique)        : {np.round(np.linalg.norm(Q_classical, axis=0), 12).tolist()}")
    print("   Qᵀ Q (classique) :")
    print(np.round(Q_classical.T @ Q_classical, 6))

**Après l'expérience**, dans ta copie : quel élément de $\mathbf{Q}^\top\mathbf{Q}$ est loin de 0 avec le procédé classique, et que dit-il des colonnes 2 et 3 de $\mathbf{Q}$ ? Pourquoi les normes restent-elles égales à 1 ? En retirant les projections sur $\mathbf{a}_3$ d'origine, que perd le procédé classique que le modifié garde ?

### Ex M12b.29 — Gram-Schmidt classique contre modifié : l'orthogonalité perdue quand les colonnes sont presque dépendantes 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer comment l'écart à l'orthogonalité des deux procédés grandit quand les colonnes deviennent presque dépendantes, et voir qu'orthogonaliser deux fois la répare.  
**Prérequis :** fiche §3, §7 et §8 ; Ex M12b.24 et M12b.28 · **Parcours :** C

Les features polynomiales $1, x, x^2, \ldots, x^d$, calculées en 50 points de $[0 ; 1]$ (la fonction fournie `polynomial_features(d)`), sont libres, mais de plus en plus **presque dépendantes** quand $d$ grandit : sur $[0 ; 1]$, $x^{10}$ et $x^{11}$ se ressemblent beaucoup (fiche §8). Écris :
- `loss_table()` : pour $d$ dans `DEGREES_29` (de 1 à 12), trois listes de 12 nombres : l'écart à l'orthogonalité (`orthogonality_loss`) de `mymath.orthogonal.gram_schmidt(..., modified=False)`, celle du procédé modifié, et celle du classique appliqué **deux fois** (`gram_schmidt(gram_schmidt(A, modified=False), modified=False)`).

La cellule suivante affiche le tableau. Lis-le, puis réponds :
- a) le plus petit degré $d$ pour lequel l'écart du classique dépasse $10^{-3}$ ;
- b) vrai ou faux : l'écart du modifié reste sous $10^{-6}$ pour tous les degrés du tableau ;
- c) vrai ou faux : l'écart du classique appliqué deux fois reste sous $10^{-12}$ pour tous les $d \le 10$ ;
- d) vrai ou faux : pour $d = 10$, l'écart du classique dépasse $0{,}5$.

Dans ta copie : de combien l'écart du classique est-il multiplié, à peu près, d'un degré au suivant ? Et celui du modifié ? Pourquoi orthogonaliser une seconde fois des colonnes déjà presque orthogonales répare-t-il presque tout ? Pour une régression polynomiale de degré 10, que choisirais-tu : les monômes et les équations normales, ou une autre base (fiche §8) ?

In [ ]:
X_POINTS_29 = np.linspace(0, 1, 50)
DEGREES_29 = list(range(1, 13))


def polynomial_features(degree):
    """The (50, degree + 1) matrix of the columns 1, x, x², …, x^degree for 50 points x of [0 ; 1]."""
    return np.vander(X_POINTS_29, degree + 1, increasing=True)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12b")


def loss_table():
    """For d in DEGREES_29: the losses of orthogonality of classical, modified and twice-classical Gram-Schmidt."""
    raise NotImplementedError("trois listes : classique, modifié, puis classique appliqué deux fois")

In [ ]:
with checked("M12b.29 tableau"):
    classical_t, modified_t, twice_t = loss_table()
    print(" d  classique   modifié    classique deux fois")
    for d, c, m, t in zip(DEGREES_29, classical_t, modified_t, twice_t):
        print(f"{d:2d}  {c:9.1e}  {m:9.1e}  {t:9.1e}")

answer_M12b_29a = ...  # a) an integer, read in the table above
answer_M12b_29b = ...  # b) True or False
answer_M12b_29c = ...  # c) True or False
answer_M12b_29d = ...  # d) True or False
for letter, answer in zip("abcd", [answer_M12b_29a, answer_M12b_29b, answer_M12b_29c, answer_M12b_29d]):
    wbm.check(f"M12b.29{letter}", answer)

### Ex M12b.30 — Projeter un nuage de ℝ³ sur un plan : les résidus orthogonaux en image 🎨 ★ ⏱️ 20 min
**Objectif :** dessiner un nuage de points, ses projetés orthogonaux sur un plan par l'origine et les segments qui les relient, et voir que ces segments sont tous perpendiculaires au plan.  
**Prérequis :** fiche §5 ; Ex M12b.25 · **Parcours :** C, E

Écris `draw_projection(ax, X, A)`, qui trace dans l'axe 3D `ax` (créé par `add_subplot(projection="3d")`), pour un nuage $\mathbf{X}$ de forme $(n, 3)$ (un point par ligne) et une matrice $\mathbf{A}$ de forme $(3, 2)$ dont les colonnes engendrent un plan $U$ :
- les points du nuage (`ax.scatter`) ;
- leurs projetés orthogonaux sur $U$, avec `mymath.orthogonal.project` (ta fonction de M12b.25), dans une autre couleur ;
- pour chaque point $\mathbf{x}$, le **segment** de $\mathbf{x}$ à son projeté $\mathbf{p}$, par un appel `ax.plot([x[0], p[0]], [x[1], p[1]], [x[2], p[2]])` (une ligne de deux points) ;
- un morceau du plan, si tu veux (`plot_surface` sur une grille de combinaisons $s\,\mathbf{a}_1 + t\,\mathbf{a}_2$).

La cellule de la figure s'en sert pour le nuage fourni ; une cellule de vérification essaie ta fonction sur deux autres nuages et deux autres plans, dans des axes neufs.

Dans ta copie : les segments paraissent-ils tous perpendiculaires au plan ? Pourquoi certains semblent-ils ne pas l'être (pense à la perspective, et tourne la figure avec `ax.view_init`) ? Quel point du nuage est le plus loin du plan, et comment le trouver par le calcul ? Remplace $\mathbf{A}$ par une autre base du même plan : que change la figure ?

In [ ]:
rng_30 = np.random.default_rng(30)
CLOUD_30 = rng_30.normal(size=(30, 3)) * np.array([2.0, 1.5, 1.0])     # 30 points of ℝ³
PLANE_30 = np.array([[1.0, 0.0], [1.0, 1.0], [0.0, 1.0]])             # its columns span a plane through the origin

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12b")


def draw_projection(ax, X, A):
    """In the 3D axis ax: the points of X (one per row), their projections on the plane spanned by the columns of A,
    and one segment from each point to its projection."""
    raise NotImplementedError("scatter des points, scatter des projetés, puis un ax.plot par segment")

In [ ]:
with checked("M12b.30 figure"):
    fig = plt.figure(figsize=(7.5, 6.5))
    ax = fig.add_subplot(projection="3d")
    try:
        draw_projection(ax, CLOUD_30, PLANE_30)
        ax.set(xlabel="x", ylabel="y", zlabel="z", title="Un nuage de ℝ³, ses projetés sur un plan, et les résidus")
        ax.view_init(elev=20, azim=-60)
    except BaseException:
        plt.close(fig)
        raise
    plt.show()

**Vérifications automatiques** (deux autres nuages, deux autres plans, dans des axes neufs).

In [ ]:
with checked("M12b.30 vérifications"):
    problems = []
    ref_30 = wbm.load_mymath("ref").orthogonal
    rng_t = np.random.default_rng(300)
    for X_t, A_t in [(rng_t.normal(size=(6, 3)), np.array([[1.0, 2.0], [0.0, 1.0], [1.0, -1.0]])),
                     (rng_t.normal(size=(4, 3)), np.array([[0.0, 1.0], [2.0, 0.0], [1.0, 1.0]]))]:
        fig_t = plt.figure()
        ax_t = fig_t.add_subplot(projection="3d")
        try:
            created = drawn(lambda: draw_projection(ax_t, X_t, A_t))
            segments = [np.column_stack(line.get_data_3d()).astype(float) for line in ax_t.lines]
        finally:
            plt.close(fig_t)
        if created:
            problems.append("draw_projection ouvre sa propre figure : dessine dans l'axe reçu")
            break
        segments = [s for s in segments if s.shape == (2, 3)]
        expected = [(x, ref_30.project(x, A_t)) for x in X_t]
        missing = [i for i, (x, p) in enumerate(expected)
                   if not any(np.allclose(s, [x, p], atol=1e-9) or np.allclose(s, [p, x], atol=1e-9) for s in segments)]
        if missing:
            problems.append(f"pour {len(X_t)} points, {len(missing)} segment(s) du point à son projeté manque(nt) ou "
                            "sont faux : un segment par point, ax.plot([x[0], p[0]], [x[1], p[1]], [x[2], p[2]])")
            break
    if problems:
        print("❌ Ex M12b.30 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M12b.30 : un segment juste par point, du point à son projeté, sur deux nuages et deux plans.")

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §3, §5, §6 et §7. Trois erreurs à trouver, un test de propriétés, puis des moindres carrés précis sur des colonnes presque dépendantes.

### Ex M12b.31 — Chasse à l'erreur : une normalisation oubliée, QQᵀ sur une base non orthonormée, R w = Qᵀy résolu dans le mauvais sens 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans du code les erreurs classiques de Gram-Schmidt, de projection et de QR, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §3, §5 et §7 · **Parcours :** C

Un collègue a écrit le code suivant ; six lignes portent une lettre et un commentaire qui dit ce qu'elles doivent calculer. Trois lignes ne calculent pas ce que dit leur commentaire. **Sans exécuter de code**, trouve-les.

```python
A = np.array([[2.0, 0.0], [1.0, 3.0], [2.0, 3.0]])      # columns a1 = (2, 1, 2), a2 = (0, 3, 3)
q1 = A[:, 0] / np.linalg.norm(A[:, 0])
q2 = A[:, 1] - (A[:, 1] @ q1) * q1           # (A) the second vector of the orthonormal basis: its norm
P = A @ A.T                                  # (B) the matrix of the projection on Vect(a1, a2): P[0, 0]
Q = np.column_stack([q1, q2 / np.linalg.norm(q2)])
v = np.array([3.0, 0.0, 0.0])
r = v - Q @ (Q.T @ v)                        # (C) the residual of v after projection on Vect(a1, a2): r[0]
R = np.triu(Q.T @ A)
y = np.array([3.0, 3.0, 6.0])
z = Q.T @ y
w = np.zeros(2)
for i in range(2):                           # (D) the least squares weights, from R w = Q^T y: w[0]
    w[i] = (z[i] - R[i, :i] @ w[:i]) / R[i, i]
loss = np.abs(Q.T @ Q - np.eye(2)).max()     # (E) the loss of orthogonality of Q: about 0
c = Q.T @ A[:, 1]                            # (F) the coordinates of a2 in the basis (q1, q2): c[1]
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) pour chacune des six lignes, de (A) à (F), la bonne valeur de ce que dit son commentaire (une liste de 6 nombres, 4 décimales).

Dans ta copie : pour chaque ligne fausse, quelle erreur a été faite, et comment la corriger ? Laquelle des trois s'exécute sans erreur **et** rend un résultat plausible, et pourquoi est-elle la plus dangereuse ?

In [ ]:
answer_M12b_31a = ...  # a) letters, as text
answer_M12b_31b = ...  # b) a list of 6 values (4 decimals)

In [ ]:
wbm.check("M12b.31a", answer_M12b_31a)
wbm.check("M12b.31b", answer_M12b_31b)

### Ex M12b.32 — Un test de propriétés pour la projection : idempotente, symétrique, résidu orthogonal 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire un test fondé sur les propriétés d'une projection orthogonale, vérifiées sur des matrices aléatoires, et voir quelle propriété attrape quelle erreur.  
**Prérequis :** fiche §5 ; Ex M12b.25 · **Parcours :** C

Les tests de `mymath` comparent `project` à une référence. On peut aussi vérifier des **propriétés** que toute projection orthogonale doit avoir (fiche §5). Écris :
- `check_projection(project, seed, trials=10)` : avec `rng = np.random.default_rng(seed)`, répète `trials` fois : tire $m$ entre 2 et 6 (`rng.integers(2, 7)`), $k$ entre 1 et $m - 1$ (`rng.integers(1, m)`), une matrice $\mathbf{A}$ de forme $(m, k)$ et deux vecteurs $\mathbf{u}$, $\mathbf{v}$ de $\mathbb{R}^m$ (loi normale). Avec $\mathbf{p}$ = `project(v, A)`, vérifie par des `assert` (avec un message), à $10^{-9}$ près : (1) **idempotence** : `project(p, A)` redonne $\mathbf{p}$ ; (2) **symétrie** : $\mathbf{P}\mathbf{u} \cdot \mathbf{v} = \mathbf{u} \cdot \mathbf{P}\mathbf{v}$ ; (3) **résidu orthogonal** : $\mathbf{A}^\top(\mathbf{v} - \mathbf{p}) = \mathbf{0}$ ;
- `test_projection()` : le test au format pytest, sans argument, qui appelle `check_projection` avec **ta** fonction (`mymath.orthogonal.project`) et une graine fixe.

La cellule fournie définit deux copies fausses : une projection **oblique** (elle envoie bien sur $\operatorname{Im}\mathbf{A}$, mais pas perpendiculairement) et $\mathbf{A}\mathbf{A}^\top\mathbf{v}$ sans Gram-Schmidt. La vérification lance ton test sur la bonne projection (la référence) et sur chaque copie fausse, pour 20 graines.

Dans ta copie : quelles propriétés attrapent la projection oblique, et laquelle la laisse passer ? Même question pour $\mathbf{A}\mathbf{A}^\top\mathbf{v}$. Pourquoi $k$ doit-il rester inférieur à $m$ (que deviendrait la projection oblique pour $k = m$) ? Une fonction fausse pourrait-elle passer les trois propriétés ?

In [ ]:
def project_oblique(v, A):                      # a copy with a bug: idempotent, but not orthogonal
    A = np.asarray(A, dtype=float)
    B = A + 0.5 * np.cos(np.arange(A.size)).reshape(A.shape)    # another basis, used to "aim" the projection
    return A @ np.linalg.solve(B.T @ A, B.T @ np.asarray(v, dtype=float))


def project_no_gram_schmidt(v, A):              # another bug: A Aᵀ v, without orthonormalizing A
    A = np.asarray(A, dtype=float)
    return A @ (A.T @ np.asarray(v, dtype=float))

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12b")


def check_projection(project, seed, trials=10):
    """Assert idempotence, symmetry and an orthogonal residual for `project`, on `trials` random cases."""
    raise NotImplementedError("tire m, k, A, u, v ; trois assert, chacun avec un message")


def test_projection():
    """The pytest test: check_projection on your project, with a fixed seed."""
    raise NotImplementedError("un appel à check_projection")

In [ ]:
with checked("M12b.32 vérifications"):
    problems = []
    ref_32 = wbm.load_mymath("ref").orthogonal

    def outcome(project_t, seed):
        """None if check_projection passes, else the message of its AssertionError."""
        try:
            check_projection(project_t, seed)
        except AssertionError as exc:
            return str(exc)
        return None

    passed = sum(outcome(ref_32.project, seed) is None for seed in range(20))
    caught_oblique = [outcome(project_oblique, seed) for seed in range(20)]
    caught_aat = [outcome(project_no_gram_schmidt, seed) for seed in range(20)]
    print(f"la bonne projection passe ton test {passed} fois sur 20 ; la projection oblique est attrapée "
          f"{sum(m is not None for m in caught_oblique)} fois sur 20 ; A Aᵀ v, "
          f"{sum(m is not None for m in caught_aat)} fois sur 20")
    if passed < 20:
        problems.append("ton test échoue sur la bonne projection : compare avec np.allclose (des flottants), et "
                        "vérifie tes formes")
    if sum(m is not None for m in caught_oblique) < 20:
        problems.append("ton test laisse passer la projection oblique : vérifie aussi la symétrie, ou que le résidu "
                        "est orthogonal aux colonnes")
    if sum(m is not None for m in caught_aat) < 20:
        problems.append("ton test laisse passer A Aᵀ v : vérifie aussi que projeter deux fois ne change rien")
    if not problems and any(not (m or "").strip() for m in caught_oblique + caught_aat):
        problems.append("ajoute un message à chaque assert (assert condition, message)")
    if not problems:
        try:
            test_projection()
        except AssertionError as exc:
            problems.append(f"test_projection échoue sur ta librairie : {exc}")
    if problems:
        print("❌ Ex M12b.32 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M12b.32 : ton test passe sur la bonne projection et attrape les deux copies fausses.")

### Ex M12b.33 — Défi : des colonnes presque dépendantes, des moindres carrés à 1e−8 près de `lstsq` par l'orthogonalisation de [X | y] 🏆 ★★★ ⏱️ 30 min
**Objectif :** résoudre des moindres carrés aussi précisément qu'une bibliothèque sur des données dont deux colonnes sont presque dépendantes, là où les équations normales et la QR naïve échouent.  
**Prérequis :** fiche §6 et §7 ; Ex M12b.24, M12b.26 et M12b.29 · **Parcours :** C

La fonction fournie `make_nearly_dependent(seed)` fabrique un tableau de 200 exemples et 6 features dont les deux dernières ne diffèrent que de $10^{-6}$ fois un bruit, et une cible presque exactement linéaire. Sur ces données, les équations normales et ta fonction `least_squares_qr` (Gram-Schmidt modifié, $\mathbf{R} = \mathbf{Q}^\top\mathbf{X}$, puis $\mathbf{Q}^\top\mathbf{y}$) s'écartent de `np.linalg.lstsq` d'environ $10^{-5}$ à $10^{-3}$ : la vérification l'affiche.

**L'objectif** : écrire `least_squares_stable(X, y)`, qui rend des poids à moins de $10^{-8}$ (le plus grand écart, coordonnée par coordonnée) de ceux de `np.linalg.lstsq` sur les trois datasets `seed` = 33, 34 et 35, **sans** `np.linalg.lstsq`, `qr`, `pinv`, `inv`, `svd`, `cholesky` ni `scipy`. Une dernière cellule enregistre si l'objectif est atteint.

Pistes (Björck, 1967) : applique le procédé **modifié** à la matrice augmentée $[\mathbf{X} \mid \mathbf{y}]$, de forme $(n, p + 1)$, en gardant les coefficients **au moment où tu les retires** : pour la colonne $k$, $r_{jk} = \mathbf{q}_j \cdot \mathbf{v}$ (le vecteur courant) pour $j < k$, puis $r_{kk} = \lVert \mathbf{v} \rVert$. La dernière colonne de ce $\mathbf{R}$ de forme $(p + 1, p + 1)$ contient, au-dessus de la diagonale, le second membre $\mathbf{z}$ ; résous $\mathbf{R}_{[:p, :p]}\,\mathbf{w} = \mathbf{z}_{[:p]}$ en remontant. Ne recalcule ni $\mathbf{R}$ par $\mathbf{Q}^\top\mathbf{X}$ ni $\mathbf{z}$ par $\mathbf{Q}^\top\mathbf{y}$ : c'est là que la QR naïve perd sa précision.

Dans ta copie : pourquoi $\mathbf{R} = \mathbf{Q}^\top\mathbf{X}$, recalculé après coup, est-il moins précis que les coefficients gardés pendant le procédé (pense à M12b.28 : $\mathbf{Q}$ n'est plus tout à fait orthonormée) ? Que représente le dernier élément diagonal de $\mathbf{R}$, $r_{p+1,p+1}$ ? Une autre voie marche-t-elle (pense à M12b.29) ?

In [ ]:
def make_nearly_dependent(seed, n=200, p=6):
    """A (n, p) design matrix whose last column is the one before plus 1e-6 times some noise, and noisy targets."""
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, p))
    X[:, p - 1] = X[:, p - 2] + 1e-6 * rng.normal(size=n)
    w_true = rng.normal(size=p)
    y = X @ w_true + 1e-6 * rng.normal(size=n)
    return X, y

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12b")


def least_squares_stable(X, y):
    """The least squares weights of y on X, within 1e-8 of np.linalg.lstsq even for nearly dependent columns."""
    raise NotImplementedError("Gram-Schmidt modifié sur [X | y], en gardant les coefficients, puis la remontée")

In [ ]:
with checked("M12b.33 défi"):
    problems, ok_33 = [], False
    ref_33 = wbm.load_mymath("ref").orthogonal
    for seed_t in (33, 34, 35):
        X_t, y_t = make_nearly_dependent(seed_t)
        w_ref = np.linalg.lstsq(X_t, y_t, rcond=None)[0]
        try:
            w_t = np.asarray(least_squares_stable(X_t.copy(), y_t.copy()), dtype=float)
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001
            problems.append(f"least_squares_stable lève {type(exc).__name__} ({exc})")
            break
        if w_t.shape != w_ref.shape:
            problems.append(f"données {seed_t} : least_squares_stable rend la forme {w_t.shape}, pas {w_ref.shape}")
            break
        gap = float(np.max(np.abs(w_t - w_ref)))
        gap_normal = float(np.max(np.abs(np.linalg.solve(X_t.T @ X_t, X_t.T @ y_t) - w_ref)))
        gap_qr = float(np.max(np.abs(ref_33.least_squares_qr(X_t, y_t) - w_ref)))
        print(f"données {seed_t} : écart à lstsq {gap:.1e} (équations normales {gap_normal:.1e}, "
              f"least_squares_qr {gap_qr:.1e})")
        if not gap <= 1e-8:
            problems.append(f"données {seed_t} : écart {gap:.1e} à lstsq, au-dessus de 1e-8")
            break
    if problems:
        print("❌ Ex M12b.33 : " + " ; ".join(problems) + ".")
    else:
        ok_33 = True
        print("✅ Ex M12b.33 : défi réussi : à moins de 1e-8 de lstsq sur les trois datasets.")
with checked("M12b.33"):
    least_squares_stable(*make_nearly_dependent(33))         # ⏳ while least_squares_stable is not written
    wbm.check("M12b.33", bool(ok_33), computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu reconnaître un produit scalaire, et calculer normes et angles avec ?
2. Sais-tu lire des coordonnées dans une base orthonormée, et reconnaître une matrice orthogonale ?
3. Sais-tu orthonormaliser une famille par Gram-Schmidt ?
4. Sais-tu calculer un complément orthogonal, une projection et une distance à un sous-espace ?
5. Sais-tu résoudre des moindres carrés par une projection, puis par QR ?

**La suite** : le chapitre 12 de workbookIA (préparation des données), en parallèle de M12c, les valeurs propres et les matrices symétriques.